# Merging and reshaping

## Learning objectives

By the end of this notebook you will be able to:

- join tables with merge, and choose the right join type
- check merges for unexpected row counts
- stack tables with concat
- reshape between wide and long formats with melt, pivot and pivot_table

## Introduction

Data usually comes in several tables: participants in one, lab results in another, site details in a
third. Merging links them through a shared key. Reshaping changes the layout: one row per participant
with a column per visit (*wide*), or one row per participant-visit (*long*). Plotting and grouping
usually want long data; reports often want wide.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

In [2]:
patients = pd.read_csv(RAW + "patients.csv")
labs = pd.read_csv(RAW + "lab_results.csv")
sites = pd.read_excel(RAW + "sites.xlsx", sheet_name="sites")
print(patients.shape, labs.shape, sites.shape)

(300, 9) (894, 7) (4, 4)


## merge

`pd.merge(left, right, on="key", how=...)` matches rows that share a key value.

| how | keeps |
|---|---|
| `"inner"` (default) | only keys found in both tables |
| `"left"` | every row of the left table; NaN where the right has no match |
| `"right"` | every row of the right table |
| `"outer"` | every key from either table |

A small example makes the differences visible:

In [3]:
a = pd.DataFrame({"id": ["P1", "P2", "P3"], "age": [34, 51, 47]})
b = pd.DataFrame({"id": ["P2", "P3", "P4"], "glucose": [7.8, 6.1, 5.2]})
for how in ["inner", "left", "right", "outer"]:
    print(how)
    print(pd.merge(a, b, on="id", how=how), "\n")

inner
   id  age  glucose
0  P2   51      7.8
1  P3   47      6.1 

left
   id  age  glucose
0  P1   34      NaN
1  P2   51      7.8
2  P3   47      6.1 

right
   id   age  glucose
0  P2  51.0      7.8
1  P3  47.0      6.1
2  P4   NaN      5.2 

outer
   id   age  glucose
0  P1  34.0      NaN
1  P2  51.0      7.8
2  P3  47.0      6.1
3  P4   NaN      5.2 



## One-to-many: patients and their visits

Each patient has several lab rows, so the patient details repeat on each visit row:

In [4]:
full = labs.merge(patients, on="patient_id", how="left", validate="many_to_one")
print(labs.shape, "->", full.shape)
full.head(4)

(894, 7) -> (894, 15)


,patient_id,visit,visit_date,glucose_mmol,haemoglobin_g_dl,systolic_bp,diastolic_bp,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
0,P0001,0,2025-04-02,4.8,14.1,119,76,Site A,M,41,66.4,156.3,No,Treatment,2025-03-31
1,P0001,1,2025-05-01,4.7,10.1,101,80,Site A,M,41,66.4,156.3,No,Treatment,2025-03-31
2,P0002,0,2025-08-19,5.6,10.9,105,80,Site B,M,83,69.5,168.0,No,Control,2025-08-19
3,P0002,1,2025-09-19,4.8,13.9,103,73,Site B,M,83,69.5,168.0,No,Control,2025-08-19


`validate="many_to_one"` makes pandas check that each patient appears only once in `patients`; if not,
it raises an error instead of silently multiplying rows. Also check the row count: after a left join on
a unique key, it should equal the left table's row count.

## Adding site information

In [5]:
full = full.merge(sites, on="site", how="left")
full[["patient_id", "visit", "site", "region", "beds"]].head()

,patient_id,visit,site,region,beds
0,P0001,0,Site A,North,120
1,P0001,1,Site A,North,120
2,P0002,0,Site B,Central,340
3,P0002,1,Site B,Central,340
4,P0002,2,Site B,Central,340


## Different key names, and the indicator column

In [6]:
register = pd.read_csv(RAW + "sample_register_raw.csv").drop_duplicates()
check = register[["sample_id", "patient_id"]].merge(
    patients[["patient_id", "site"]].rename(columns={"patient_id": "pid"}),
    left_on="patient_id", right_on="pid", how="left", indicator=True)
check["_merge"].value_counts()

_merge
both          420
left_only       0
right_only      0
Name: count, dtype: int64

`indicator=True` adds `_merge`, saying whether each row matched (`both`) or not (`left_only`). It is the
quickest way to find IDs that do not link, a common data-quality check.

## concat: stacking tables

`concat` puts tables on top of each other (same columns) or side by side.

In [7]:
batch1 = pd.DataFrame({"sample": ["S1", "S2"], "value": [5.1, 6.3]})
batch2 = pd.DataFrame({"sample": ["S3", "S4"], "value": [4.8, 7.0]})
pd.concat([batch1, batch2], ignore_index=True)

,sample,value
0,S1,5.1
1,S2,6.3
2,S3,4.8
3,S4,7.0


In [8]:
pd.concat([batch1.assign(batch=1), batch2.assign(batch=2)], ignore_index=True)

,sample,value,batch
0,S1,5.1,1
1,S2,6.3,1
2,S3,4.8,2
3,S4,7.0,2


Reading many files with the same layout and stacking them is a common pattern:

```python
from pathlib import Path
frames = [pd.read_csv(f).assign(source=f.name) for f in Path("data").glob("run_*.csv")]
combined = pd.concat(frames, ignore_index=True)
```

## Wide and long data

In [9]:
wide = labs.pivot(index="patient_id", columns="visit", values="systolic_bp")
wide.columns = [f"sbp_visit{v}" for v in wide.columns]
wide.head()

,sbp_visit0,sbp_visit1,sbp_visit2,sbp_visit3
patient_id,,,,
P0001,119.0,101.0,NaN,NaN
P0002,105.0,103.0,135.0,NaN
P0003,140.0,121.0,NaN,NaN
P0004,105.0,106.0,105.0,NaN
P0005,114.0,126.0,134.0,130.0


That is **wide**: one row per patient, one column per visit. `pivot` requires each (index, column) pair
to be unique; when it is not, use `pivot_table` with an `aggfunc`.

`melt` turns wide back into **long**:

In [10]:
long = wide.reset_index().melt(id_vars="patient_id", var_name="visit", value_name="systolic_bp")
long = long.dropna().sort_values(["patient_id", "visit"])
long.head(6)

,patient_id,visit,systolic_bp
0,P0001,sbp_visit0,119.0
300,P0001,sbp_visit1,101.0
1,P0002,sbp_visit0,105.0
301,P0002,sbp_visit1,103.0
601,P0002,sbp_visit2,135.0
2,P0003,sbp_visit0,140.0


## Several measurements: long format for plotting

Lab results have three measurements per visit. Melting them into one column lets you group or plot all
of them in one step:

In [11]:
measures = labs.melt(id_vars=["patient_id", "visit"],
                     value_vars=["glucose_mmol", "haemoglobin_g_dl", "systolic_bp"],
                     var_name="test", value_name="value")
print(measures.shape)
measures.groupby("test")["value"].describe().round(2)

(2682, 4)


,count,mean,std,min,25%,50%,75%,max
test,,,,,,,,
glucose_mmol,894.0,6.05,1.08,3.0,5.3,6.0,6.8,10.1
haemoglobin_g_dl,894.0,13.22,1.52,8.2,12.2,13.2,14.2,17.7
systolic_bp,894.0,127.45,14.77,86.0,118.0,128.0,137.0,175.0


## Gene expression: from a matrix to tidy data

Expression data come as a matrix (samples × genes). Melting gives one row per sample-gene, which makes
per-gene comparisons a simple groupby:

In [12]:
expr = pd.read_csv(RAW + "gene_expression.tsv", sep="\t")
tidy = expr.melt(id_vars=["sample", "condition"], var_name="gene", value_name="expression")
print(expr.shape, "->", tidy.shape)

by_gene = tidy.pivot_table(index="gene", columns="condition", values="expression", aggfunc="mean")
by_gene["difference"] = by_gene["treated"] - by_gene["control"]
by_gene.sort_values("difference", key=abs, ascending=False).head(5).round(2)

(24, 22) -> (480, 4)


condition,control,treated,difference
gene,,,
GENE05,7.62,9.75,2.13
GENE04,7.69,9.64,1.95
GENE02,8.09,9.67,1.58
GENE01,8.11,9.22,1.11
GENE18,9.01,8.04,-0.97


The genes with the largest mean differences between conditions are candidates for a closer look. A
difference in means is not yet evidence: it needs the spread and the number of samples (a statistical
test), which is beyond this course.

## stack and unstack

These move levels between the index and the columns, and are the index-based relatives of melt and pivot:

In [13]:
two_level = patients.groupby(["site", "group"])["age"].mean().round(1)
print(two_level.unstack())
print(two_level.unstack().stack().head(4))

group   Control  Treatment
site                      
Site A     52.1       46.7
Site B     55.9       56.2
Site C     48.4       52.9
Site D     48.8       47.5
site    group    
Site A  Control      52.1
        Treatment    46.7
Site B  Control      55.9
        Treatment    56.2
dtype: float64


## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)